In [1]:
import unstructured
print("unstructured ok")

unstructured ok


In [3]:
pip install -Uq langchain_chroma


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install -Uq langchain langchain-community langchain-openai


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install -Uq python_dotenv


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import json
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

/Users/gaziwasifakram/multi_modal_rag/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.


False

In [17]:
def partition_document(file_path: str):
    """Extract elements from pdf using unstructured"""
    print(f"Partitioning document: {file_path}")

    elements = partition_pdf(
        filename = file_path,
        strategy = "hi_res",
        infer_table_structure = True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload = True
    )

    print(f"Extracted {len(elements)} elements.")
    return elements

In [19]:
file_path = "attention-is-all-you-need.pdf"
elements = partition_document(file_path=file_path)

Partitioning document: attention-is-all-you-need.pdf


Loading weights: 100%|██████████████████████████████████████████████████████████████████████| 367/367 [00:00<00:00, 1216.73it/s, Materializing param=model.query_position_embeddings.weight]


Extracted 215 elements.


In [21]:
set([str(type(element)) for element in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Formula'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [41]:
elements[49].to_dict()

{'type': 'Image',
 'element_id': '6f5f5a92fc9fea5732681d6b267bf0a8',
 'text': 'Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention Add & Norm Masked Multi-Head Attention Add & Norm Feed Forward Add & Norm Multi-Head Attention Nx Positional Encoding O° Positional @ OY Encoding Input Output Embedding Embedding Inputs Outputs (shifted right)',
 'metadata': {'coordinates': {'points': ((np.float64(545.9972222222221),
     np.float64(200.00555555555542)),
    (np.float64(545.9972222222221), np.float64(1095.6055555555556)),
    (np.float64(1153.997222222222), np.float64(1095.6055555555556)),
    (np.float64(1153.997222222222), np.float64(200.00555555555542))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-02-10T17:36:20',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'filename': 'attention-is-all-you-need.pdf'}}

In [51]:
def create_chunks_by_title(elements):
    """Create chunks usinf title based strategy"""
    print("Creating chunks....")
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars = 2400,
        combine_text_under_n_chars = 500
    )

    print(f"Created {len(chunks)} chunks.")
    return chunks

In [53]:
chunks = create_chunks_by_title(elements=elements)

Creating chunks....
Created 25 chunks.


In [55]:
chunks

In [61]:
chunks[4].to_dict()

{'type': 'CompositeElement',
 'element_id': '7f2a82c0-d015-4987-85aa-546adce86d86',
 'text': '3 Model Architecture\n\nMost competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations (x1,...,xn) to a sequence of continuous representations z = (z1,...,zn). Given z, the decoder then generates an output sequence (y1,...,ym) of symbols one element at a time. At each step the model is auto-regressive [10], consuming the previously generated symbols as additional input when generating the next.\n\n2\n\nOutput Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention Add & Norm Masked Multi-Head Attention Add & Norm Feed Forward Add & Norm Multi-Head Attention Nx Positional Encoding O° Positional @ OY Encoding Input Output Embedding Embedding Inputs Outputs (shifted right)\n\nFigure 1: The Transformer - model architecture.\n\nThe Transformer follows this overall architecture using 

In [97]:
chunks[5].metadata.orig_elements[3].to_dict()

IndexError: list index out of range

In [103]:
chunks[4].metadata.orig_elements[3].to_dict()

{'type': 'Image',
 'element_id': '6f5f5a92fc9fea5732681d6b267bf0a8',
 'text': 'Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention Add & Norm Masked Multi-Head Attention Add & Norm Feed Forward Add & Norm Multi-Head Attention Nx Positional Encoding O° Positional @ OY Encoding Input Output Embedding Embedding Inputs Outputs (shifted right)',
 'metadata': {'coordinates': {'points': ((np.float64(545.9972222222221),
     np.float64(200.00555555555542)),
    (np.float64(545.9972222222221), np.float64(1095.6055555555556)),
    (np.float64(1153.997222222222), np.float64(1095.6055555555556)),
    (np.float64(1153.997222222222), np.float64(200.00555555555542))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-02-10T17:36:20',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'filename': 'attention-is-all-you-need.pdf'}}

In [107]:
def separate_content_types(chunk):
    """Analyze what type of contents are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    if hasattr(chunk,'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)

            if element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    content_data['types'] = list(set(content_data['types']))
    return content_data

In [113]:
def create_ai_enhanced_summary(text:str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for hybrid chunks"""

    try:
        llm = ChatOpenAI(
            model = "gpt-4o",
            temperature = 0,
            # api_key = OPEN AI API KEY
        )

        prompt_text = f"""You are creating a searchable description for document content retreival.

        Content to analyze:
        Text content:
        {text}
        """

        if tables:
            prompt_text += "Tables:\n"
            for i,table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""
        message_content = [{"type":"text", "text":promp_text}]

        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        return response.content
    except Exception as e:
        print(f"AI summary failed: {e}")
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Containes {len(tables)} tables]"
        if images:
            summary += f" [Contains {len(images)} images]"
        return summary

In [137]:
def summarise_chunk(chunks):
    """Process all chunks with AI summarizer. Only add summary for hybrid chunks"""
    print("Processing chunks with AI summarizer")

    langchain_documents = []
    total_chunks = len(chunks)

    for i,chunk in enumerate(chunks):
        current_chunk = i+1
        print(f"Processing chunk {current_chunk}/{total_chunks}")
        content_data = separate_content_types(chunk)

        print(f"Types found: {content_data['types']}")
        print(f"Tables found: {len(content_data['tables'])}, Images found: {len(content_data['images'])}")

        if content_data['tables'] or content_data['images']:
            print(f" Creating AI summary for hubrid chunk")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                print("AI summary successfully created")
            except Exception as e:
                print(f"AI summary failed: {e}")
                enhanched_content = content_data['text']
        else:
            print("Using raw text(no tables or images")
            enhanced_content = content_data['text']

        doc = Document(
            page_content = enhanced_content,
            metadata={
                "original_content": json.dumps(
                    {
                        "raw_text": content_data['text'],
                        "tables_html": content_data['tables'],
                        "images_base64": content_data['images']
                    }
                )
            }
        )
        langchain_documents.append(doc)
    print(f"Processed {len(langchain_documents)} chunks")
    return langchain_documents
                
            

In [127]:
def create_vector_store(documents, persist_directory="db/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("Creating embeddings and storing in vector db")
    embedding_model = OpenAIEmbeddings(model = "text-embedding-3-small",
                                    #    api_key = OPEN AI API KEY
                                      )

    print("----Creating vector store----")
    vectorstore = Chroma.from_documents(
        documents = documents,
        embedding = embedding_model,
        persist_directory = persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("Vector store created")
    return vectorstore
    

In [139]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the full rag ingestion pipeline"""
    print("Starting RAG ingestion pipeline")
    print("="*50)

    elements = partition_document(pdf_path)
    chunks = create_chunks_by_title(elements)
    summarised_chunks = summarise_chunk(chunks)
    db = create_vector_store(summarised_chunks, "db/chroma_db")
    print("Pipeline completed successfully")
    return db

In [141]:
db = run_complete_ingestion_pipeline("attention-is-all-you-need.pdf")

Starting RAG ingestion pipeline
Partitioning document: attention-is-all-you-need.pdf
Extracted 215 elements.
Creating chunks....
Created 25 chunks.
Processing chunks with AI summarizer
Processing chunk 1/25
Types found: ['text']
Tables found: 0, Images found: 0
Using raw text(no tables or images
Processing chunk 2/25
Types found: ['text']
Tables found: 0, Images found: 0
Using raw text(no tables or images
Processing chunk 3/25
Types found: ['text']
Tables found: 0, Images found: 0
Using raw text(no tables or images
Processing chunk 4/25
Types found: ['text']
Tables found: 0, Images found: 0
Using raw text(no tables or images
Processing chunk 5/25
Types found: ['image', 'text']
Tables found: 0, Images found: 1
 Creating AI summary for hubrid chunk
AI summary failed: name 'promp_text' is not defined
AI summary successfully created
Processing chunk 6/25
Types found: ['text']
Tables found: 0, Images found: 0
Using raw text(no tables or images
Processing chunk 7/25
Types found: ['image', 't

In [145]:

query = "How many attention heads does the Transformer use, and what is the dimension of each head? "

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
       
        llm = ChatOpenAI(
            model = "gpt-4o",
            temperature = 0,
            # api_key = Use explicit api key if needed 
        )
        
        
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents. If answer is found add citation."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

The Transformer uses 8 attention heads, and the dimension of each head is 64. This is specified in Document 3, which states: "In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64."
